# ARIA - LITE

ARIA Lite is a lightweight GraphRAG-based biomedical research assistant focused on breast cancer AI literature.

The project combines two retrieval paradigms:

1. Semantic Retrieval
   Dense vector embeddings are used to retrieve papers semantically related to a user query.

2. Graph-Based Retrieval
   Biomedical entities such as genes and drugs are extracted from papers and represented as relationships in a lightweight knowledge graph.

By combining these two approaches, the system aims to provide more grounded and explainable retrieval compared to traditional vector-only RAG systems.

The project is intentionally scoped for rapid iteration and learning:
- ~300-500 PubMed papers
- Abstract-only corpus
- Lightweight graph construction
- Citation-grounded responses

Core technologies:
- PubMed / Entrez API
- SciSpacy
- Sentence Transformers
- FAISS
- Python + Google Colab

End Goal:
Build a small but functional biomedical GraphRAG system capable of retrieving relevant breast cancer AI papers and generating grounded answers with PMID citations.

# 9_gradio_app.ipynb
Goal: Build an interactive biomedical GraphRAG application using:
- Hybrid Retrieval
- Knowledge Graph Retrieval
- FAISS Vector Search
- Local LLM QA
- Interactive Graph Visualization

In [1]:
!pip install gradio networkx faiss-cpu sentence-transformers transformers matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 59.3 MB/s eta 0:00:00


In [2]:
import os
import json
import pickle
import faiss
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt

import gradio as gr

from sentence_transformers import SentenceTransformer
from transformers import pipeline

In [3]:
PROJECT_ROOT = "/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite"

GRAPH_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "aria_lite_graph_v2.pkl")

PAPERS_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "clean_papers.json")

FAISS_INDEX_PATH = os.path.join(PROJECT_ROOT, "data", "indices", "faiss_index.bin")

PMIDS_PATH = os.path.join(PROJECT_ROOT, "data", "indices", "pmids.pkl")

TEXTS_PATH = os.path.join(PROJECT_ROOT, "data", "indices", "texts.pkl")

In [4]:
with open(PAPERS_PATH, "r") as f:
    papers = json.load(f)

with open(GRAPH_PATH, "rb") as f:
    G = pickle.load(f)

index = faiss.read_index(FAISS_INDEX_PATH)

with open(PMIDS_PATH, "rb") as f:
    pmids = pickle.load(f)

with open(TEXTS_PATH, "rb") as f:
    texts = pickle.load(f)

print("Loaded everything")
print("Graph nodes:", G.number_of_nodes())
print("Graph edges:", G.number_of_edges())
print("FAISS vectors:", index.ntotal)

Loaded everything
Graph nodes: 1108
Graph edges: 19839
FAISS vectors: 475


In [5]:
paper_lookup = {}

for p in papers:
    if isinstance(p, dict) and "pmid" in p:
        paper_lookup[p["pmid"]] = p

In [6]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
def vector_search(query, k=5):

    q_emb = model.encode([query]).astype("float32")

    scores, idxs = index.search(q_emb, k)

    results = []

    for score, idx in zip(scores[0], idxs[0]):

        if idx < 0:
            continue

        pmid = pmids[idx]

        results.append((pmid, float(score)))

    return results

In [8]:
def graph_expand(pmid, hops=1):

    nodes = set([pmid])
    frontier = set([pmid])

    for _ in range(hops):

        new_frontier = set()

        for n in frontier:
            if n in G:
                new_frontier.update(list(G.neighbors(n)))

        nodes.update(new_frontier)
        frontier = new_frontier

    return list(nodes)

In [9]:
def hybrid_retrieve(query, top_k=5):

    vec_results = vector_search(query, k=top_k)

    enriched = []

    for pmid, score in vec_results:

        neighbors = graph_expand(pmid, hops=1)

        enriched.append({
            "pmid": pmid,
            "score": score,
            "neighbors": neighbors
        })

    return enriched

In [10]:
llm = pipeline(
    "text-generation",
    model="microsoft/Phi-3-mini-4k-instruct",
    device_map="auto"
)

print("LLM loaded")

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

LLM loaded


In [11]:
def build_context(results):

    context = ""

    for r in results[:3]:

        pmid = r["pmid"]

        if pmid not in paper_lookup:
            continue

        p = paper_lookup[pmid]

        context += f"""
                    PMID: {pmid}
                    Title and abstract: : {p.get('text','')}
                    --------------------------------------------------
                    """

    return context

In [12]:
def generate_answer(query, results):

    context = build_context(results)

    prompt = f"""
You are a biomedical research assistant. If not enough information is found, say exactly: "Insufficient evidence in retrieved papers."

Answer the question ONLY using the provided research context.


QUESTION:
{query}

RESEARCH CONTEXT:
{context}

ANSWER:
"""

    out = llm(
        prompt,
        max_new_tokens=300,
        do_sample=False
    )

    return out[0]["generated_text"]

In [13]:
def plot_graph(pmids, max_nodes=30):

    subG = nx.Graph()

    for p in pmids:

        if p in G:
            subG.add_node(p)

            neighbors = list(G.neighbors(p))[:5]

            for nb in neighbors:
                subG.add_node(nb)
                subG.add_edge(p, nb)

    # limit clutter
    if subG.number_of_nodes() > max_nodes:
        subG = subG.subgraph(list(subG.nodes())[:max_nodes])

    plt.figure(figsize=(10, 7))

    pos = nx.spring_layout(subG, seed=42)

    colors = []

    for n in subG.nodes():
        if n in pmids:
            colors.append("red")        # seed papers
        else:
            colors.append("lightblue")  # neighbors

    nx.draw(
        subG,
        pos,
        node_color=colors,
        node_size=500,
        edge_color="gray",
        with_labels=True,
        font_size=8
    )

    return plt

In [14]:
def run_pipeline(query):

    results = hybrid_retrieve(query)

    pmids = [r["pmid"] for r in results]

    answer = generate_answer(query, results)

    graph_fig = plot_graph(pmids)

    return answer, graph_fig

In [15]:
with gr.Blocks() as app:

    gr.Markdown("# 🧬 ARIA Lite — GraphRAG Explorer")

    query = gr.Textbox(label="Biomedical Query")

    btn = gr.Button("Search")

    answer_box = gr.Textbox(
        label="🧠 Answer",
        lines=10
    )

    graph_box = gr.Plot(
        label="🕸 Knowledge Graph"
    )

    btn.click(
        fn=run_pipeline,
        inputs=query,
        outputs=[answer_box, graph_box]
    )

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://97cd2ad9cb6ceeaa93.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
